# FE747 Data Analytics in Finance
## Strategy Diagnostic Tool

**Instructions:**
1. Run Cell 1 — mounts Drive (run once per session)
2. Run Cell 2 — select file, click **▶ Load & Run Diagnostics**

> Charts saved to `diagnostics/` in your Drive folder.

In [ ]:
# @title ⚙️ Cell 1 — Drive Mount (run once) { display-mode: "form" }
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH        = '/content/drive/My Drive/MSFin_Python/final_project/'
SIMULATIONS_PATH  = os.path.join(DRIVE_PATH, 'simulations/')
DIAGNOSTICS_PATH  = os.path.join(DRIVE_PATH, 'diagnostics/')
os.makedirs(DIAGNOSTICS_PATH, exist_ok=True)

STATE_COLORS = {
    'Aggressive': '#27ae60',
    'Neutral':    '#bdc3c7',
    'Conflicted': '#f39c12',
    'Defensive':  '#c0392b',
}
C1       = '#2980b9'
C2       = '#8e44ad'
S1_LABEL = 'Strategy 1'
S2_LABEL = 'Strategy 2'

print("\u2705 Drive mounted")
print(f"\u2705 Simulations: {SIMULATIONS_PATH}")
print(f"\u2705 Diagnostics: {DIAGNOSTICS_PATH}")
print("\nRun Cell 2 to load a simulation file and generate charts.")

In [ ]:
# @title 📊 Cell 2 — Load & Run Diagnostics { display-mode: "form" }

# ── File selector ─────────────────────────────────────────────────────────────
sim_files = sorted([
    f for f in os.listdir(SIMULATIONS_PATH)
    if f.startswith('sim_') and f.endswith('.csv')
])

if not sim_files:
    print("\u274c No simulation files found in simulations/ folder.")
else:
    print(f"Found {len(sim_files)} simulation file(s).")

file_selector = widgets.Dropdown(
    options=sim_files,
    description='Select File:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
load_btn = widgets.Button(
    description='\u25b6 Load & Run Diagnostics',
    button_style='success',
    layout=widgets.Layout(width='220px')
)
load_out = widgets.Output()

display(widgets.HBox([file_selector, load_btn]))
display(load_out)

# ── Helpers ───────────────────────────────────────────────────────────────────
def shade_states(ax, dates, states):
    prev_state = None
    prev_date  = dates[0]
    for date, state in zip(dates, states):
        if pd.isna(state):
            continue
        if state != prev_state:
            if prev_state is not None and prev_state in STATE_COLORS:
                ax.axvspan(prev_date, date,
                           color=STATE_COLORS[prev_state], alpha=0.15, linewidth=0)
            prev_state = state
            prev_date  = date
    if prev_state in STATE_COLORS:
        ax.axvspan(prev_date, dates[-1],
                   color=STATE_COLORS[prev_state], alpha=0.15, linewidth=0)

def run_charts(df):
    pct_fmt = FuncFormatter(lambda x, _: f'{x:.0%}')

    # ── CHART 1 — Relative Cumulative Wealth ──────────────────────────────────
    rel1 = (df['strat1_cum_strat'] - df['strat1_cum_bmk']).where(
             lambda x: x.abs() > 1e-10, other=0.0)
    rel2 = (df['strat2_cum_strat'] - df['strat2_cum_bmk']).where(
             lambda x: x.abs() > 1e-10, other=0.0)

    fig, ax = plt.subplots(figsize=(13, 4))
    y_max = max(rel1.abs().max(), rel2.abs().max())
    if y_max < 1e-10:
        ax.text(0.5, 0.5, 'No differentiation from benchmark\n(all allocations = 50/50)',
                ha='center', va='center', fontsize=12, color='gray',
                transform=ax.transAxes)
    else:
        ax.plot(df.index, rel1, color=C1, lw=1.8, label=f'{S1_LABEL} vs Bmk')
        ax.plot(df.index, rel2, color=C2, lw=1.8, label=f'{S2_LABEL} vs Bmk')
        ax.axhline(0, color='black', lw=0.8, ls='--')
        ax.fill_between(df.index, rel1, 0, where=(rel1 >= 0), color=C1, alpha=0.08)
        ax.fill_between(df.index, rel1, 0, where=(rel1 < 0),  color=C1, alpha=0.08)
        ax.fill_between(df.index, rel2, 0, where=(rel2 >= 0), color=C2, alpha=0.08)
        ax.fill_between(df.index, rel2, 0, where=(rel2 < 0),  color=C2, alpha=0.08)
        ax.annotate(f'{rel1.iloc[-1]:+.2%}',
                    xy=(df.index[-1], rel1.iloc[-1]),
                    xytext=(6, 0), textcoords='offset points',
                    fontsize=8, color=C1, fontweight='bold', va='center',
                    clip_on=False)
        ax.annotate(f'{rel2.iloc[-1]:+.2%}',
                    xy=(df.index[-1], rel2.iloc[-1]),
                    xytext=(6, 0), textcoords='offset points',
                    fontsize=8, color=C2, fontweight='bold', va='center',
                    clip_on=False)
        ax.legend(loc='upper left', fontsize=8)
    ax.set_title('Relative Cumulative Wealth (Strategy \u2212 Benchmark)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Wealth Difference (base 1.0)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:+.2%}'))
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()

    plt.show()
    print("\u2705 Chart 1: Relative Cumulative Wealth")

    # ── CHART 2 — Probability Series + Active State ───────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, hspace=0.35)
    axes[0].set_facecolor('#eaf4fb')
    axes[1].set_facecolor('#f9ebf5')

    for ax, label, p_up_col, p_dd_col, state_col in [
        (axes[0], S1_LABEL,
         'strat1_UP_logit_pct', 'strat1_DD_logit_pct', 'strat1_active_state'),
        (axes[1], S2_LABEL,
         'strat2_UP_logit_pct', 'strat2_DD_logit_pct', 'strat2_active_state'),
    ]:
        shade_states(ax, df.index, df[state_col])
        ax.plot(df.index, df[p_up_col], color='#27ae60', lw=1.2, label='P(Upside)')
        ax.plot(df.index, df[p_dd_col], color='#c0392b', lw=1.2, label='P(Drawdown)')
        ax.set_title(f'{label} \u2014 Daily Probabilities & Active State',
                     fontsize=10, fontweight='bold')
        ax.set_ylabel('Probability')
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(pct_fmt)
        ax.grid(axis='y', alpha=0.3)
        patches = [mpatches.Patch(color=v, alpha=0.4, label=k)
                   for k, v in STATE_COLORS.items()]
        line_handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles=line_handles + patches,
                  loc='upper left', fontsize=7, ncol=3)
        state_counts = df[state_col].value_counts()
        total        = state_counts.sum()
        summary      = '  |  '.join([
            f"{s}: {state_counts.get(s, 0) / total:.0%}"
            for s in ['Aggressive', 'Neutral', 'Conflicted', 'Defensive']
        ])
        ax.text(0.01, 0.02, f"State mix: {summary}",
                transform=ax.transAxes, fontsize=7,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    plt.show()
    print("\u2705 Chart 2: Probability Series + Active State")

    # ── CHART 3 — Simplified Confusion Matrices ───────────────────────────────
    C_TN = '#f4a7a7'; C_FP = '#c0392b'; C_FN = '#a8d5b0'; C_TP = '#27ae60'

    eq_ret  = df['strat1_EQ_ret'].dropna()
    cum_fwd = (1 + eq_ret).rolling(63).apply(np.prod, raw=True).shift(-63)
    fwd_ret = cum_fwd - 1
    realized_up = (fwd_ret >=  0.05).astype(float)
    realized_dd = (fwd_ret <= -0.05).astype(float)

    fig, axes = plt.subplots(2, 2, figsize=(12, 6))
    fig.suptitle('Simplified Confusion Matrices (\u00b15% realized threshold)',
                 fontsize=11, fontweight='bold')

    def plot_cm(ax, predicted, actual, title):
        predicted = predicted.reindex(actual.index).dropna()
        actual    = actual.reindex(predicted.index).dropna()
        idx       = predicted.index.intersection(actual.index)
        predicted = predicted.loc[idx]
        actual    = actual.loc[idx]
        if len(predicted) == 0:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                    fontsize=10, color='gray', transform=ax.transAxes)
            ax.axis('off'); return
        tp = int(((predicted==1)&(actual==1)).sum())
        fp = int(((predicted==1)&(actual==0)).sum())
        fn = int(((predicted==0)&(actual==1)).sum())
        tn = int(((predicted==0)&(actual==0)).sum())
        n_obs = len(predicted)
        acc   = (tp+tn)/n_obs
        prec0 = tn/(tn+fn) if (tn+fn)>0 else 0
        rec0  = tn/(tn+fp) if (tn+fp)>0 else 0
        prec1 = tp/(tp+fp) if (tp+fp)>0 else 0
        rec1  = tp/(tp+fn) if (tp+fn)>0 else 0
        cell_colours = [[C_TN,C_FP],[C_FN,C_TP]]
        cell_values  = [[tn,fp],[fn,tp]]
        cell_tags    = [['TN','FP'],['FN','TP']]
        ax.set_xlim(0,2); ax.set_ylim(0,2.6); ax.invert_yaxis()
        for r in range(2):
            for c in range(2):
                rect = mpatches.FancyBboxPatch(
                    (c+0.04,r+0.04), 0.92, 0.82,
                    boxstyle='round,pad=0.02',
                    facecolor=cell_colours[r][c], edgecolor='white', lw=2)
                ax.add_patch(rect)
                count = cell_values[r][c]
                tag   = cell_tags[r][c]
                ax.text(c+0.50, r+0.22, f'{tag}  {count}',
                        ha='center', va='center',
                        fontsize=11, fontweight='bold', color='white')
                ax.text(c+0.50, r+0.50, f'{count/n_obs*100:.1f}% obs',
                        ha='center', va='center', fontsize=8, color='white')
                ax.text(c+0.50, r+0.74, 'Pred 0' if c==0 else 'Pred 1',
                        ha='center', va='center', fontsize=7, color='white', style='italic')
        stats = (f'Acc: {acc:.1%}  Base: {actual.mean():.1%}  N={n_obs:,}\n'
                 f'Prec  0:{prec0:.1%}  1:{prec1:.1%}\n'
                 f'Recall 0:{rec0:.1%}  1:{rec1:.1%}')
        ax.text(1.0, 2.30, stats, ha='center', va='center',
                fontsize=7.5, family='monospace',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#d6d6d6',
                          edgecolor='#888', lw=1))
        ax.set_xticks([]); ax.set_yticks([0.5,1.5])
        ax.set_yticklabels(['Act 0 (No)','Act 1 (Yes)'], fontsize=7)
        ax.set_title(title, fontweight='bold', fontsize=9, pad=4)
        for spine in ax.spines.values(): spine.set_visible(False)
        ax.tick_params(length=0)

    for col, (label, p_up_col, p_dd_col) in enumerate([
        (S1_LABEL, 'strat1_UP_logit_pct', 'strat1_DD_logit_pct'),
        (S2_LABEL, 'strat2_UP_logit_pct', 'strat2_DD_logit_pct'),
    ]):
        up_signal = (df[p_up_col].reindex(fwd_ret.index) >=
                     df[p_up_col].median()).astype(float)
        dd_signal = (df[p_dd_col].reindex(fwd_ret.index) >=
                     df[p_dd_col].median()).astype(float)
        plot_cm(axes[0,col], up_signal, realized_up,
                f'{label} \u2014 Upside (\u2265+5%)')
        plot_cm(axes[1,col], dd_signal, realized_dd,
                f'{label} \u2014 Drawdown (\u2264-5%)')

    fig.tight_layout()

    plt.show()
    print("\u2705 Chart 3: Confusion Matrices")
    print("\n\u2705 All diagnostics complete \u2014 saved to diagnostics/ folder")

# ── Load callback ─────────────────────────────────────────────────────────────
def on_load(btn):
    with load_out:
        clear_output()
        try:
            fpath = os.path.join(SIMULATIONS_PATH, file_selector.value)
            df = pd.read_csv(fpath, index_col=0, parse_dates=True)
            # Validate
            required = ['strat1_cum_strat','strat1_cum_bmk',
                        'strat2_cum_strat','strat2_cum_bmk',
                        'strat1_UP_logit_pct','strat1_DD_logit_pct',
                        'strat1_active_state','strat2_active_state',
                        'strat1_EQ_ret']
            missing = [c for c in required if c not in df.columns]
            if missing:
                print(f"\u274c Invalid file — missing columns: {missing}")
                return
            print(f"\u2705 Loaded: {file_selector.value}  ({len(df):,} rows)")
            print(f"   Strategy 1: {df['strat1_EQ_fund'].iloc[0]} / {df['strat1_FI_fund'].iloc[0]}")
            print(f"   Strategy 2: {df['strat2_EQ_fund'].iloc[0]} / {df['strat2_FI_fund'].iloc[0]}")
            print(f"   Window:     {df.index[0].strftime('%b %Y')} \u2192 {df.index[-1].strftime('%b %Y')}")
            print("\nGenerating charts...\n")
            run_charts(df)
        except Exception as e:
            print(f"\u274c Load failed: {e}")

load_btn.on_click(on_load)